# 03 — 1D-CNN (derinlik penceresi)
Her derinlik, aynı kuyudaki üst/alt komşularıyla birlikte (`window` satır) modele girer. Soru: bağlam görmek tahmini iyileştiriyor mu?

In [ ]:
# --- Kurulum: Colab'da repoyu klonlar/günceller, yerelde repo köküne geçer ---
import os, sys, pathlib
REPO_URL = "https://github.com/aysenursimsk/force2020-litoloji.git"   # ← kendi repo adresiniz

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/force2020-litoloji").exists():
        !git clone -q {REPO_URL} /content/force2020-litoloji
    %cd /content/force2020-litoloji
    !git pull -q

root = pathlib.Path.cwd()
if not (root / "src").exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
%load_ext autoreload
%autoreload 2

In [ ]:
KIM = ""          # ← adınız: deney kaydında görünür
N_WELLS = None    # hızlı deneme için ör. 20 (sadece ilk 20 eğitim kuyusu); final için None

In [ ]:
from src import veri, modeller as M, egitim as E, degerlendir as D

d = veri.hazirla(n_wells=N_WELLS)
d.ozet()
cw = E.sinif_agirliklari(d.y_tr)

WINDOW = 33   # tek sayı: ortadaki satır tahmin edilir
ds_tr = veri.pencere_dataset(d.X_tr, d.y_tr, d.w_tr, WINDOW, shuffle=True, class_weight=cw)
ds_va = veri.pencere_dataset(d.X_va, d.y_va, d.w_va, WINDOW)
ds_te = veri.pencere_dataset(d.X_te, d.y_te, d.w_te, WINDOW)

In [ ]:
ayar = dict(window=WINDOW, filters=(64, 128), kernel=5, dropout=0.3, l2=1e-4, lr=1e-3)
model = M.build_cnn1d(d.X_tr.shape[1], **ayar)
model.summary()
hist = E.egit_ds(model, ds_tr, ds_va, epochs=20)
D.ogrenme_egrisi(hist, "1D-CNN")

In [ ]:
p = model.predict(ds_te, verbose=0).argmax(1)
s = D.skorlar(d.y_te, p); print(s)
D.rapor(d.y_te, p)
D.confusion(d.y_te, p, "1D-CNN")
D.kaydet("cnn_temel", "1D-CNN", ayar | {"class_weight": True}, s, kim=KIM, epochs=len(hist.history["loss"]))

In [ ]:
# Tahminleri bir test kuyusunda gerçekle karşılaştır
D.kuyu_grafigi(d.te, d.te.WELL.iloc[0], y_pred=p)